# 0 - Construction de la base de données

## Importation des modules

In [1]:
# Modules de base
import os
import json
import pandas as pd
import sys
import yaml

# Ajout du chemin
sys.path.append('..')

# Importation des modules ad hoc
from dashboard_template_database.builders.schema import SchemaBuilder
from dashboard_template_database.builders.tables import DuckdbTablesBuilder
from dashboard_template_database.storage.loader import Loader
from dashboard_template_database.operations.updater_v2 import DatabaseUpdaterV2
from dashboard_template_database.operations.deleter_v2 import DatabaseDeleterV2

# Chargement du fichier de configurations
with open("../config.yaml") as file:
    config = yaml.safe_load(file)

# Chargement du fichier de parmaètres
with open("../parameters/labels.json") as file:
    labels = json.load(file)


ModuleNotFoundError: No module named 'sqlparse'

## Importation des données

In [ ]:
# Initialisation du loader
loader = Loader()
# Importation des données
df_origin = loader.load(filepath=os.path.join('../', config['INPUT_DATA']))
# Conversion en datetime
df_origin['date'] = pd.to_datetime(df_origin['date'])
df_origin.head()

,indicator,country,date,value,kind,horizon,week,model,training
0,Gross Domestic Product,France,1960-04-01,0.375710,observed,NaN,NaN,NaN,NaN
1,Gross Domestic Product,France,1960-07-01,0.748561,observed,NaN,NaN,NaN,NaN
2,Gross Domestic Product,France,1960-10-01,1.185218,observed,NaN,NaN,NaN,NaN
3,Gross Domestic Product,France,1961-01-01,1.608374,observed,NaN,NaN,NaN,NaN
4,Gross Domestic Product,France,1961-04-01,1.600329,observed,NaN,NaN,NaN,NaN


## 1 - Construction du schéma

### Initialisation de la classe

In [ ]:
# Initialisation du schéma
schema_builder = SchemaBuilder(df=df_origin, categorical_threshold=config['THRESHOLD'])

### Construction des méta-données

In [ ]:
# Construction du jeu de métadonnées
df_metadata = schema_builder.create_metadata_table(column_labels=labels)

df_metadata.head()

2025-03-22 17:15:34,861 - INFO - Successfully extracted meta-data from column 'indicator'
2025-03-22 17:15:34,895 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2025-03-22 17:15:34,895 - INFO - Successfully extracted meta-data from column 'country'
2025-03-22 17:15:34,928 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2025-03-22 17:15:34,928 - INFO - Successfully extracted meta-data from column 'date'
2025-03-22 17:15:34,928 - INFO - Successfully extracted meta-data from column 'value'
2025-03-22 17:15:34,928 - INFO - Successfully extracted meta-data from column 'kind'
2025-03-22 17:15:34,960 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2025-03-22 17:15:34,961 - INFO - Successfully extracted meta-data from column 'horizon'
2025-03-22 17:15:

,name,label,python_type,sql_type,is_categorical
0,country,Country,object,VARCHAR,True
1,date,Date,datetime64[ns],TIMESTAMP,False
2,horizon,Horizon,float64,DOUBLE,False
3,indicator,Indicator,object,VARCHAR,True
4,kind,Kind,object,VARCHAR,True


### Construction des tables de dimensions

In [ ]:
# Construction des types de dimensions
dimension_tables = schema_builder.create_dimension_tables(column_labels=labels)
dimension_tables['indicator'].head()

2025-03-22 17:15:35,015 - INFO - Successfully extracted meta-data from column 'indicator'
2025-03-22 17:15:35,047 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2025-03-22 17:15:35,057 - INFO - Successfully extracted meta-data from column 'country'
2025-03-22 17:15:35,084 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2025-03-22 17:15:35,084 - INFO - Successfully extracted meta-data from column 'date'
2025-03-22 17:15:35,084 - INFO - Successfully extracted meta-data from column 'value'
2025-03-22 17:15:35,084 - INFO - Successfully extracted meta-data from column 'kind'
2025-03-22 17:15:35,113 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2025-03-22 17:15:35,113 - INFO - Successfully extracted meta-data from column 'horizon'
2025-03-22 17:15:

,value,label
0,0,Gross Domestic Product
1,1,Private Consumption


### Construction de la table d'information

In [ ]:
# Construction de la table d'informations
df_fact = schema_builder.create_fact_table(column_labels=labels)
df_fact.head()

C:\Users\bolli\OneDrive\Documents\COD Code\dashboard-template-database\dashboard_template_database\builders\schema.py:197: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df_fact[column] = self.df_fact[column].replace(dict_label_value)
2025-03-22 17:15:35,534 - INFO - Successfully replace modalities by ids in column 'country'
C:\Users\bolli\OneDrive\Documents\COD Code\dashboard-template-database\dashboard_template_database\builders\schema.py:197: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df_fact[column] = self.df_fact[

,indicator,country,date,value,kind,horizon,week,model,training
0,0,0,1960-04-01,0.375710,0,NaN,NaN,0.0,0.0
1,0,0,1960-07-01,0.748561,0,NaN,NaN,0.0,0.0
2,0,0,1960-10-01,1.185218,0,NaN,NaN,0.0,0.0
3,0,0,1961-01-01,1.608374,0,NaN,NaN,0.0,0.0
4,0,0,1961-04-01,1.600329,0,NaN,NaN,0.0,0.0


### Création de l'ensemble des tables

In [ ]:
# Création de l'ensemble des tables du schéma
df_metadata, dimension_tables, df_fact = schema_builder.build(column_labels=labels)

2025-03-22 17:15:36,183 - INFO - Successfully extracted meta-data from column 'indicator'
2025-03-22 17:15:36,273 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2025-03-22 17:15:36,273 - INFO - Successfully extracted meta-data from column 'country'
2025-03-22 17:15:36,307 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2025-03-22 17:15:36,307 - INFO - Successfully extracted meta-data from column 'date'
2025-03-22 17:15:36,307 - INFO - Successfully extracted meta-data from column 'value'
2025-03-22 17:15:36,307 - INFO - Successfully extracted meta-data from column 'kind'
2025-03-22 17:15:36,342 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2025-03-22 17:15:36,342 - INFO - Successfully extracted meta-data from column 'horizon'
2025-03-22 17:15:

## 2 - Construction de la base de données

### Initialisation du builder

In [ ]:
# Initialisation du builder
builder = DuckdbTablesBuilder(df=df_origin, categorical_threshold=config['THRESHOLD'], path=os.path.join('../', config['OUTPUT_DATA']))

### Création du schéma

In [ ]:
# Construction du schéma duckDB
builder.build_duckdb_schema()

2025-03-22 17:15:38,239 - INFO - Successfully extracted meta-data from column 'indicator'
2025-03-22 17:15:38,304 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2025-03-22 17:15:38,305 - INFO - Successfully extracted meta-data from column 'country'
2025-03-22 17:15:38,340 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2025-03-22 17:15:38,342 - INFO - Successfully extracted meta-data from column 'date'
2025-03-22 17:15:38,343 - INFO - Successfully extracted meta-data from column 'value'
2025-03-22 17:15:38,344 - INFO - Successfully extracted meta-data from column 'kind'
2025-03-22 17:15:38,382 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2025-03-22 17:15:38,383 - INFO - Successfully extracted meta-data from column 'horizon'
2025-03-22 17:15:

### Affichage du schéma

In [ ]:
# Affichage du schéma
builder.display_schema()

2025-03-22 17:15:41,522 - INFO - 
 Created Tables:
2025-03-22 17:15:41,523 - INFO - 
 dim_country Structure:
2025-03-22 17:15:41,526 - INFO -   value: BIGINT
2025-03-22 17:15:41,527 - INFO -   label: VARCHAR
2025-03-22 17:15:41,528 - INFO - 
 dim_indicator Structure:
2025-03-22 17:15:41,530 - INFO -   value: BIGINT
2025-03-22 17:15:41,532 - INFO -   label: VARCHAR
2025-03-22 17:15:41,534 - INFO - 
 dim_kind Structure:
2025-03-22 17:15:41,536 - INFO -   value: BIGINT
2025-03-22 17:15:41,536 - INFO -   label: VARCHAR
2025-03-22 17:15:41,537 - INFO - 
 dim_model Structure:
2025-03-22 17:15:41,540 - INFO -   value: BIGINT
2025-03-22 17:15:41,542 - INFO -   label: VARCHAR
2025-03-22 17:15:41,542 - INFO - 
 dim_training Structure:
2025-03-22 17:15:41,546 - INFO -   value: BIGINT
2025-03-22 17:15:41,547 - INFO -   label: VARCHAR
2025-03-22 17:15:41,549 - INFO - 
 fact_table Structure:
2025-03-22 17:15:41,551 - INFO -   indicator: BIGINT
2025-03-22 17:15:41,554 - INFO -   country: BIGINT
2025-

### Exemple de requête

In [ ]:
# Requête de la table d'information
print(builder.conn.execute("SELECT * FROM dim_model").fetchall())

[(1, 'ElasticNetCV'), (2, 'ExtraTrees'), (3, 'LassoCV'), (4, 'RandomForest'), (5, 'RidgeCV'), (6, 'XGBStandard'), (0, None)]


## 3- Mise à jour de la base de données

### Construction des données de mise à jour

In [ ]:
# Création de nouvelles données à insérer/mettre à jour
update_data = pd.DataFrame({
    'indicator': ['Gross Domestic Product', 'Private Consumption', 'Gross Domestic Product'],
    'country': ['Germany', 'Germany', 'France'],
    'date': pd.to_datetime(['2023-01-01', '2023-01-01', '2023-01-01']),
    'value': [100.5, 75.2, 150.8],
    'kind': ['forecast', 'forecast', 'forecast'],
    'horizon': [4.0, 4.0, 4.0],
    'week': [1.0, 1.0, 1.0],
    'model': ['XGBStandard', 'RandomForest', 'LassoCV'],
    'training': ['training', 'training', 'training']
})

update_data.head()

### Exécution de la mise à jour

In [ ]:
# Initialisation de l'updater
updater = DatabaseUpdaterV2(
    connection=builder.conn,
    categorical_threshold=config['THRESHOLD'],
    enable_validation=True
)

# Affichage du nombre de lignes avant la mise à jour
print(f"Nombre de lignes dans la fact table avant mise à jour:")
print(builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0])

# Mise à jour de la base de données avec les nouvelles données
success = updater.update_database(
    update_df=update_data,
    check_duplicates_db=True,
    check_duplicates_update=True,
    keep='last',
    use_batch_processing=False,
    use_transaction=True
)

# Affichage
if success:
    print("✓ Mise à jour effectuée avec succès")
    print(f"\nNombre de lignes dans la fact table après mise à jour:")
    print(builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0])
else:
    print("✗ Échec de la mise à jour")

# Vérification des données ajoutées pour 2023
# Création de la requête
query = """
SELECT 
    f.date,
    c.label as country,
    i.label as indicator,
    k.label as kind,
    f.value,
    f.horizon
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
LEFT JOIN dim_indicator i ON f.indicator = i.value
LEFT JOIN dim_kind k ON f.kind = k.value
WHERE YEAR(f.date) = 2023
ORDER BY f.date, c.label, i.label
"""
# Exécution de la requête
result_df = builder.conn.execute(query).fetchdf()

# Affichage
print(f"Données ajoutées pour l'année 2023 ({len(result_df)} lignes):")

result_df.head()

## 4 - Suppression des données

### Analyse des données avant suppression

In [ ]:
# Comptage des lignes avant suppression
total_rows = builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Nombre total de lignes: {total_rows}")

# Comptage des lignes pour la France
france_query = """
SELECT COUNT(*) 
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
WHERE c.label = 'France'
"""
france_rows = builder.conn.execute(france_query).fetchone()[0]
print(f"Nombre de lignes pour la France: {france_rows}")

# Répartition par pays
country_distribution = builder.conn.execute("""
SELECT c.label as country, COUNT(*) as count
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
GROUP BY c.label
ORDER BY count DESC
""").fetchdf()

print("\nRépartition par pays:")
country_distribution.head()

### Exécution de la suppression

In [ ]:
# Initialisation du deleter
deleter = DatabaseDeleterV2(
    connection=builder.conn,
    categorical_threshold=config['THRESHOLD'],
    enable_validation=True,
    auto_cleanup=True
)

# Analyse de l'impact de la suppression
# Récupération de l'ID de la France depuis dim_country
france_id_query = "SELECT value FROM dim_country WHERE label = 'France'"
france_id = builder.conn.execute(france_id_query).fetchone()[0]

# Construction du filtre pour la France
filters = [('country', '=', france_id)]

# Analyse de l'impact
impact_report = deleter.get_deletion_impact(filters=filters)

# Affichage
print("Rapport d'impact de la suppression:")
print(f"  - Lignes affectées: {impact_report.get('rows_affected', 0)}")
print(f"  - Colonnes affectées: {impact_report.get('columns_affected', [])}")
print(f"  - Tables de dimension affectées: {impact_report.get('dimension_tables_affected', [])}")
print(f"  - Index affectés: {impact_report.get('indexes_affected', [])}")

if impact_report.get('warnings'):
    print("\nAvertissements:")
    for warning in impact_report['warnings']:
        print(f"  - {warning}")

if impact_report.get('recommendations'):
    print("\nRecommandations:")
    for recommendation in impact_report['recommendations']:
        print(f"  - {recommendation}")

# Suppression des lignes relatives à la France
deleted_rows = deleter.delete_rows(
    filters=filters,
    use_transaction=True,
    perform_cleanup=True
)

# Affichage
if deleted_rows >= 0:
    print(f"✓ Suppression effectuée avec succès")
    print(f"  Nombre de lignes supprimées: {deleted_rows}")
else:
    print("✗ Échec de la suppression")

### Vérification après suppression

In [ ]:
# Vérification après suppression
total_rows_after = builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Nombre total de lignes après suppression: {total_rows_after}")
print(f"Différence: {total_rows - total_rows_after} lignes supprimées\n")

# Vérification qu'il n'y a plus de lignes pour la France
france_rows_after = builder.conn.execute(france_query).fetchone()[0]
print(f"Nombre de lignes pour la France après suppression: {france_rows_after}")

# Nouvelle répartition par pays
country_distribution_after = builder.conn.execute("""
SELECT c.label as country, COUNT(*) as count
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
GROUP BY c.label
ORDER BY count DESC
""").fetchdf()

# Affichage
print("\nNouvelle répartition par pays:")
country_distribution_after.head()